# Praktikum 3 : klasifikasi rating dari teks ulasan

In [1]:
# Langkah 1 - Load Data
import numpy as np
import pandas as pd
df = pd.read_csv('../../dataset_pbl/reviews.csv')
df.head()

,review_id,cafe_id,author_name,rating,text,published_at,likes,scraped_at
0,02d3c3c3-df6f-4533-9785-bfc63996c796,02069a80-3a5e-4a4a-8a48-39cf0b41b534,Dayat. Art,5.0,"Rekomendasi buat tempat ngopi, pisang goreng n...",2026-10-17T10:18:43.848451+00:00,0,2026-10-17T10:18:43.845715+00:00
1,0ad747c7-6b5a-46d7-99a8-03cf9c24404d,02069a80-3a5e-4a4a-8a48-39cf0b41b534,Hafidh Arfi,5.0,"Teh susux enak, ngupi sambil melihat lalu lalang",2026-10-17T10:18:43.849451+00:00,0,2026-10-17T10:18:43.845715+00:00
2,0e704df4-9664-4134-8681-6920d4b07fd9,02069a80-3a5e-4a4a-8a48-39cf0b41b534,Lutfi Jhn23,5.0,tempat santai dan ena buat nongkrong rame rame,2026-10-17T10:18:43.849451+00:00,0,2026-10-17T10:18:43.845715+00:00
3,289fa293-bb96-4809-adf0-ae48951a3b2a,02069a80-3a5e-4a4a-8a48-39cf0b41b534,Naufal Ridhwan Aly,5.0,Resmi menamatkan Warkop Djayantie,2026-10-17T10:18:43.849451+00:00,0,2026-10-17T10:18:43.845715+00:00
4,2bba471d-2346-4722-b90b-1aab2cebac93,02069a80-3a5e-4a4a-8a48-39cf0b41b534,Mira Suhartanti,5.0,Pisang goreng nya JUARA !\nharus coba pokoknya,2026-10-17T10:18:43.849451+00:00,0,2026-10-17T10:18:43.845715+00:00


In [2]:
# Langkah 2a - Memilih kolom dan mengganti nama
df = df[['rating', 'text']].rename(columns={'rating': 'Labels', 'text': 'Teks'})
df.head()

,Labels,Teks
0,5.0,"Rekomendasi buat tempat ngopi, pisang goreng n..."
1,5.0,"Teh susux enak, ngupi sambil melihat lalu lalang"
2,5.0,tempat santai dan ena buat nongkrong rame rame
3,5.0,Resmi menamatkan Warkop Djayantie
4,5.0,Pisang goreng nya JUARA !\nharus coba pokoknya


In [3]:
# Langkah 2b - Inspeksi Data
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())
print('\n')

# Cek Statistik Deskriptif
print(df.describe())

Labels
5.0    6321
4.0     692
1.0     576
3.0     239
2.0     182
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 8010 entries, 0 to 8009
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Labels  8010 non-null   float64
 1   Teks    8010 non-null   str    
dtypes: float64(1), str(1)
memory usage: 963.0 KB
None


            Labels
count  8010.000000
mean      4.498127
std       1.144153
min       1.000000
25%       5.000000
50%       5.000000
75%       5.000000
max       5.000000


In [4]:
# Langkah 2c - Encoding label rating 1 sampai 5
new_labels = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
df['Labels'] = df['Labels'].map(new_labels)
df.head()

,Labels,Teks
0,4,"Rekomendasi buat tempat ngopi, pisang goreng n..."
1,4,"Teh susux enak, ngupi sambil melihat lalu lalang"
2,4,tempat santai dan ena buat nongkrong rame rame
3,4,Resmi menamatkan Warkop Djayantie
4,4,Pisang goreng nya JUARA !\nharus coba pokoknya


In [5]:
# Langkah 2d - Pisahkan Fitur dengan Label
X = df['Teks'].values
y = df['Labels'].values

In [6]:
# Langkah 3 - Ekstraksi Fitur
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

X = df['Teks'].to_numpy(dtype=str)
y = df['Labels'].to_numpy(dtype=int)

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test
# Mengapa hanya transform? Alasan yang sama dengan kasus pada percobaan ke-3
# Kita tidak menginginkan model mengetahui paramter yang digunakan oleh CountVectorizer untuk fitting data X_train
# Sehingga, data testing dapat tetap menjadi data yang asing bagi model nantinya
X_test = bow.transform(X_test)


# Cek fitur dari proses CountVectorizer.
print(f'Jumlah fitur: {len(bow.get_feature_names_out())}')
print(f'Dimensi data X_train: {X_train.shape}')

Jumlah fitur: 10459
Dimensi data X_train: (6408, 10459)


In [7]:
# Langkah 4 - Training dan Evaluasi Model
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data training
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data training
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.8852996254681648
Hasil akurasi data test: 0.8420724094881398
